___
<h3> Read the data structure from the JSON file, that is a result of raw file preprocessing with the <a href='data_preprocessing.ipynb'>pre-processing notebook </a>.</h3>

___

In [ ]:
import os
import json

data_processed_folder = "data_processed"
json_file = "pages_with_tables.json"
json_path = os.path.join(data_processed_folder, json_file)

with open(json_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents")

all_pages = [
    {
        "document_name": doc["document_name"],
        "page_num": page.get("page_num"),
        "text": page.get("text", ""),
        "tables": page.get("tables", [])
    }
    for doc in documents
    for page in doc.get("pages", [])
]

print(f"Loaded {len(all_pages)} total pages")

pages_with_tables = [p for p in all_pages if p["tables"]]
print(f"Pages with tables: {len(pages_with_tables)}")

all_tables = [
    {
        "document_name": page["document_name"],
        "page_num": page["page_num"],
        "table_index": idx + 1,
        "table": table
    }
    for page in all_pages
    for idx, table in enumerate(page.get("tables", []))
]

print(f"Total parsed tables: {len(all_tables)}")

____
<h3> We build signatures to figure out how many unique tables we have, to determine the strategy for grouping the data. This also helps us to develop a strategy on creating a SQL Schema that covers all the tables from all documents </h3>

____

<h3> Fixing anomalies manually detected - as there are some columns that have no names and the library enumerates them with col1, col2 etc. By analyzing those tables, it was identified that headers have duplicate rows, and that empty cell effectively means the repetition of the previous first non-empty cell.</h3>

In [ ]:
import re

COL_PATTERN = re.compile(r'^col\d+$')

def fill_col_headers(headers):
    """Replace col{n} with last_real_name_1, last_real_name_2, etc."""
    result = []
    last_real = None
    col_count = 0
    for h in headers:
        if COL_PATTERN.match(h):
            col_count += 1
            result.append(f"{last_real}_{col_count}" if last_real else h)
        else:
            last_real = h
            col_count = 0
            result.append(h)
    return result

patched_tables = 0

for doc in documents:
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            headers = table.get("headers", [])
            if not any(COL_PATTERN.match(h) for h in headers):
                continue

            old_headers = list(headers)
            new_headers = fill_col_headers(old_headers)
            table["headers"] = new_headers

            rename_map = {old: new for old, new in zip(old_headers, new_headers) if old != new}

            for row in table.get("rows", []):
                for old_key, new_key in rename_map.items():
                    if old_key in row:
                        row[new_key] = row.pop(old_key)

            patched_tables += 1

___
<h3> Analyzing headers and fixing anomalies </h3>

<h3> The point of the cell below is to figure out (for the analysis purposes) unique tables, and patters of column name repetitions </h3>

___

In [ ]:
import hashlib
import re

def signature_id(sig_tuple):
    return hashlib.sha1("||".join(sig_tuple).encode("utf-8")).hexdigest()

def header_signature(headers: list) -> tuple:
    """Returns a normalized tuple suitable for use as a dict key / hash input."""
    result = []
    for idx, h in enumerate(headers):
        h = re.sub(r"\s+", "_", h.strip().lower())
        h = re.sub(r"[^a-z0-9_]+", "", h)
        result.append(h if h else f"col_{idx+1}")
    return tuple(result)

# Collect unique table structures
unique_signatures = {}

for doc in documents:
    for page in doc.get("pages", []):
        for table in page.get("tables", []):
            raw_headers = table.get("headers", [])
            sig = header_signature(raw_headers)
            if sig:
                sid = signature_id(sig)
                if sid not in unique_signatures:
                    unique_signatures[sid] = raw_headers

# Convert to list for iteration
catalog_list = [{"raw_headers_example": headers} for headers in unique_signatures.values()]

In [ ]:
for c in catalog_list:
    print(f"Raw Headers Example: {c['raw_headers_example']}")

___
<h3> The cell below is a result of analyzing the previous headers, finding commmon parts and making a decision to merge the data from similar headers. In reality, such decision would be brought by the domain expert, and this is a great example where domain experts enrich AI capabilities. </h3>

___

In [ ]:
import sqlite3
import os
import re
import json

# ── Classification ────────────────────────────────────────────────────────────
CCT_CODES = {'827','830','835','840','842','850','865','927','930','935','940','950','965'}

def normalize_headers(headers):
    seen = {}
    new = []
    for h in headers:
        if h not in seen:
            seen[h] = 0
            new.append(h)
        else:
            seen[h] += 1
            new.append(f"{h}_{seen[h]}")
    return new

def classify_table(table):
    headers = normalize_headers(table.get("headers", []))
    hset = set(headers)
    rows = table.get("rows", [])

    if "commercial_product_name" in hset:
        return "ordering_data", {"headers": headers, "rows": rows}

    cct_col = next((h for h in headers if h in CCT_CODES), None)
    if cct_col and "lm" in hset:
        new_rows = []
        for row in rows:
            new_row = {}
            for h in headers:
                if h in CCT_CODES:
                    new_row["tc_condition"] = row.get(h, "")
                    new_row["cct_code"] = cct_col
                else:
                    new_row[h] = row.get(h, "")
            new_rows.append(new_row)
        return "photometric_by_cct", {"headers": headers, "rows": new_rows}

    if "flux_" in hset or "efficacy_" in hset:
        if any(h.startswith("tc_c") or h.startswith("tcase_c") for h in headers):
            return "temperature_tuning", {"headers": headers, "rows": rows}
        if "i_ma" in hset:
            return "current_tuning", {"headers": headers, "rows": rows}

    if any(h.startswith("l70") or h.startswith("l80") or h.startswith("l90") for h in headers):
        return "lumen_maintenance", {"headers": headers, "rows": rows}

    if "parameter" in hset and "unit" in hset:
        if "typ" in hset and "min" in hset:
            return "parameter_min_typ_max", table
        if "min" in hset:
            return "parameter_min_max", table
        if "nominal" in hset:
            return "parameter_nominal_life_max", table
        if "value" in hset:
            return "parameter_value", table
        return "parameter_other", table

    if "specification_item" in hset:
        return "wiring_spec", table

    if "application" in hset:
        return "application", table

    return "unknown", table


# ── Helpers ───────────────────────────────────────────────────────────────────
def to_real(v):
    try:
        if v is None or v == "":
            return None
        return float(str(v).replace(">", "").strip())
    except:
        return None

def split_products(s):
    return [p.strip() for p in s.split(",") if p.strip()]

SPEC_FAMILIES = {
    "parameter_min_typ_max",
    "parameter_min_max",
    "parameter_nominal_life_max",
    "parameter_value",
    "parameter_other",
    "wiring_spec",
    "application",
}
PERFORMANCE_FAMILIES = {
    "temperature_tuning",
    "current_tuning",
    "photometric_by_cct",
}


# ── Schema ────────────────────────────────────────────────────────────────────
db_path = "data_processed/products.db"
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE product (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    commercial_name TEXT,
    eoc TEXT,
    nc_12 TEXT,
    eprel_registration TEXT,
    box_quantity INTEGER,
    document_name TEXT,
    page_num INTEGER,
    source_table_family TEXT
);

CREATE TABLE product_spec (
    spec_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER,
    spec_name TEXT,
    unit TEXT,
    value TEXT,
    min_value TEXT,
    nominal TEXT,
    max_value TEXT,
    condition TEXT,
    life_value TEXT,
    raw_json TEXT,
    document_name TEXT,
    page_num INTEGER,
    source_table_family TEXT
);

CREATE TABLE product_performance (
    perf_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER,
    color_code TEXT,
    operation_point TEXT,
    temperature_case TEXT,
    luminous_flux REAL,
    efficacy REAL,
    input_current_ma REAL,
    tc_numeric REAL,
    extra_label TEXT,
    document_name TEXT,
    page_num INTEGER,
    source_table_family TEXT
);

CREATE TABLE product_lifetime (
    life_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_id INTEGER,
    operation_point TEXT,
    temperature_case TEXT,
    l70_b50 TEXT,
    l70_b20 TEXT,
    l70_b10 TEXT,
    l80_b50 TEXT,
    l80_b20 TEXT,
    l80_b10 TEXT,
    l90_b50 TEXT,
    l90_b20 TEXT,
    l90_b10 TEXT,
    document_name TEXT,
    page_num INTEGER,
    source_table_family TEXT
);

CREATE INDEX idx_product_name ON product(commercial_name);
CREATE INDEX idx_spec_product ON product_spec(product_id);
CREATE INDEX idx_perf_product ON product_performance(product_id);
CREATE INDEX idx_life_product ON product_lifetime(product_id);
""")

conn.commit()


# ── Product helper ────────────────────────────────────────────────────────────
product_cache = {}

def get_or_create_product(
    cur,
    name,
    eoc=None,
    nc_12=None,
    eprel=None,
    box_qty=None,
    document_name=None,
    page_num=None,
    source_table_family=None,
):
    if name in product_cache:
        return product_cache[name]

    cur.execute("SELECT product_id FROM product WHERE commercial_name = ?", (name,))
    row = cur.fetchone()
    if row:
        product_cache[name] = row[0]
        return row[0]

    cur.execute("""
        INSERT INTO product (
            commercial_name, eoc, nc_12, eprel_registration, box_quantity,
            document_name, page_num, source_table_family
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        name, eoc, nc_12, eprel, box_qty,
        document_name, page_num, source_table_family
    ))

    pid = cur.lastrowid
    product_cache[name] = pid
    return pid


# ── Ingestion ─────────────────────────────────────────────────────────────────
stats = {"products": 0, "specs": 0, "performance": 0, "lifetime": 0, "skipped": 0}

for doc in documents:
    document_name = doc.get("document_name")

    for page in doc.get("pages", []):
        page_num = page.get("page_num")

        for raw_table in page.get("tables", []):
            family, table = classify_table(raw_table)
            rows = table.get("rows", [])

            # ── PRODUCT ────────────────────────────────────────
            if family == "ordering_data":
                for row in rows:
                    name = row.get("commercial_product_name")
                    if not name:
                        continue

                    get_or_create_product(
                        cur,
                        name,
                        eoc=row.get("eoc"),
                        nc_12=row.get("12nc"),
                        eprel=row.get("eprel_registration_") or row.get("eprel_registration"),
                        box_qty=row.get("box_quantity"),
                        document_name=document_name,
                        page_num=page_num,
                        source_table_family=family,
                    )
                    stats["products"] += 1

            # ── SPEC ───────────────────────────────────────────
            elif family in SPEC_FAMILIES:
                for row in rows:
                    products = split_products(row.get("related_products", ""))
                    if not products:
                        continue

                    spec_name = row.get("parameter") or row.get("specification_item") or "application"

                    if family == "application":
                        unit = None
                        value = row.get("application_1") or row.get("application")
                        min_val = None
                        typ_val = None
                        max_val = None
                        life_val = None
                        condition = None
                    else:
                        unit = row.get("unit")
                        value = row.get("value")
                        min_val = row.get("min")
                        typ_val = row.get("typ") or row.get("nominal")
                        max_val = row.get("max")
                        life_val = row.get("life")
                        condition = row.get("condition")

                    for p in products:
                        pid = get_or_create_product(
                            cur,
                            p,
                            document_name=document_name,
                            page_num=page_num,
                            source_table_family=family,
                        )

                        cur.execute("""
                            INSERT INTO product_spec (
                                product_id, spec_name, unit, value, min_value, nominal,
                                max_value, condition, life_value, raw_json,
                                document_name, page_num, source_table_family
                            )
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """, (
                            pid, spec_name, unit, value, min_val, typ_val,
                            max_val, condition, life_val, json.dumps(row),
                            document_name, page_num, family
                        ))
                        stats["specs"] += 1

            # ── PERFORMANCE ────────────────────────────────────
            elif family in PERFORMANCE_FAMILIES:
                for row in rows:
                    products = split_products(row.get("related_products", ""))
                    if not products:
                        continue

                    for p in products:
                        pid = get_or_create_product(
                            cur,
                            p,
                            document_name=document_name,
                            page_num=page_num,
                            source_table_family=family,
                        )

                        cur.execute("""
                            INSERT INTO product_performance (
                                product_id, color_code, operation_point, temperature_case,
                                luminous_flux, efficacy, input_current_ma, tc_numeric,
                                extra_label, document_name, page_num, source_table_family
                            )
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """, (
                            pid,
                            row.get("cct_code"),
                            row.get("operation_point"),
                            row.get("tc_condition") or row.get("tc_c") or row.get("tcase_c"),
                            to_real(row.get("lm") or row.get("flux_")),
                            to_real(row.get("lmw") or row.get("efficacy_")),
                            to_real(row.get("i_ma")),
                            to_real(row.get("tc_c") or row.get("tcase_c")),
                            row.get("operation_window"),
                            document_name,
                            page_num,
                            family
                        ))
                        stats["performance"] += 1

            # ── LIFETIME ───────────────────────────────────────
            elif family == "lumen_maintenance":
                for row in rows:
                    products = split_products(row.get("related_products", ""))
                    if not products:
                        continue

                    for p in products:
                        pid = get_or_create_product(
                            cur,
                            p,
                            document_name=document_name,
                            page_num=page_num,
                            source_table_family=family,
                        )

                        cur.execute("""
                            INSERT INTO product_lifetime (
                                product_id, operation_point, temperature_case,
                                l70_b50, l70_b20, l70_b10,
                                l80_b50, l80_b20, l80_b10,
                                l90_b50, l90_b20, l90_b10,
                                document_name, page_num, source_table_family
                            )
                            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                        """, (
                            pid,
                            row.get("operation_point"),
                            row.get("lumen_maintenancebrx_1000_hours"),
                            row.get("l70"),
                            row.get("l70_1"),
                            row.get("l70_2"),
                            row.get("l80"),
                            row.get("l80_1"),
                            row.get("l80_2"),
                            row.get("l90"),
                            row.get("l90_1"),
                            row.get("l90_2"),
                            document_name,
                            page_num,
                            family
                        ))
                        stats["lifetime"] += 1

            else:
                stats["skipped"] += 1

conn.commit()

# optional quick checks
for table_name in ["product", "product_spec", "product_performance", "product_lifetime"]:
    cur.execute(f"SELECT COUNT(*) FROM {table_name}")
    print(table_name, cur.fetchone()[0])

conn.close()

print("DONE")
print(stats)

___
<h3> We store to the local index purpose information about the products, and textual summary without tabular forms </h3>

___

<h3> Using FAISS (Facebook AI Similarity Search, library and implementation for local indexing) </h3>

In [ ]:
import random
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

# Set random seeds for deterministic behavior
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

model = SentenceTransformer('all-MiniLM-L6-v2')
model.eval()  # Set to evaluation mode for deterministic inference

In [ ]:
import os
import json
import pickle
import faiss
import numpy as np

# Load extracted records
with open("data_processed/purpose_design_extractions.json", "r", encoding="utf-8") as f:
    extracted_records = json.load(f)

# Prepare documents and metadata
documents = []
metadata = []

for r in extracted_records:
    text = r.get("extracted_text", "").strip()
    if text:
        documents.append(text)
        metadata.append({
            "document_name": r.get("document_name", "unknown_document"),
            "related_products": r.get("related_products", []),
            "related_products_csv": ", ".join(r.get("related_products", [])),
        })

print(f"Loaded {len(documents)} documents")

# Generate embeddings using the model loaded in previous cell
print("Generating embeddings...")
embeddings = model.encode(
    documents, 
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=False,  # No normalization for L2 distance
    batch_size=32  # Consistent batch size
)
embeddings = np.array(embeddings).astype('float32')

print(f"Embeddings shape: {embeddings.shape}")

# Create FAISS index (L2 distance)
embed_dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(embed_dim)

# Add embeddings to index
faiss_index.add(embeddings)

print(f"Added {faiss_index.ntotal} vectors to FAISS index")

# Persist both metadata and FAISS index
persist_dir = "data_processed/faiss_index"
os.makedirs(persist_dir, exist_ok=True)

# Save FAISS index
faiss_index_path = os.path.join(persist_dir, "faiss.index")
faiss.write_index(faiss_index, faiss_index_path)

# Save metadata and documents
metadata_path = os.path.join(persist_dir, "metadata.pkl")
with open(metadata_path, "wb") as f:
    pickle.dump({"documents": documents, "metadata": metadata}, f)

# Also save as JSON for human readability
metadata_json_path = os.path.join(persist_dir, "metadata.json")
with open(metadata_json_path, "w", encoding="utf-8") as f:
    json.dump({"documents": documents, "metadata": metadata}, f, indent=2, ensure_ascii=False)

print(f"Saved FAISS index to {faiss_index_path}")
print(f"Saved metadata to {metadata_path} and {metadata_json_path}")
print(f"Persisted FAISS index to {persist_dir}")

In [ ]:
# -----------------------------
# Quick query test using pure FAISS
# -----------------------------

question = (
    "Which products are best suited for retail applications? "
    "Return product names and short reasoning based on purpose/design context."
)

print("\nQuery:\n")
print(question)

# Generate query embedding
query_embedding = model.encode([question], convert_to_numpy=True).astype('float32')

# Search top-k similar documents
k = 5
distances, indices = faiss_index.search(query_embedding, k)

# Display results
print("\nTop source matches:\n")
for i, (dist, idx) in enumerate(zip(distances[0], indices[0]), start=1):
    print(f"{i}. document_name: {metadata[idx]['document_name']}")
    print(f"   related_products: {metadata[idx]['related_products_csv']}")
    print(f"   distance: {dist:.4f}")
    print(f"   text preview: {documents[idx][:200]}...")
    print()